# 範例 13：企業合約法律風險預警與審查分流系統

**學習目標**：
- 了解非工程師（法務、法遵、上班族）如何把日常審查的合約歷史資料轉化為機器學習模型
- 理解特徵（X）與目標標籤（y：低/中/高風險）的概念
- 使用 scikit-learn Pipeline 封裝前處理與隨機森林（Random Forest）演算法
- 評估模型並輸出完整的預測模型（`model.joblib`），為後續製作 Tkinter 桌面 App 做好準備

## 0. 環境準備與繁體中文字型設定
為防止 Matplotlib 在繪製圖表時出現中文方塊亂碼（Tofu）或負號異常，我們先載入支援繁體中文的字型。

In [ ]:
import os
import glob
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# 嘗試載入 mplfonts 或系統中文字型
try:
    import mplfonts
    from matplotlib import font_manager
    from mplfonts import use_font
    for f in glob.glob(os.path.join(os.path.dirname(mplfonts.__file__), "fonts", "*.[ot]tf")):
        font_manager.fontManager.addfont(f)
    use_font("Noto Sans CJK SC")
except Exception:
    plt.rcParams["font.sans-serif"] = ["Arial Unicode MS", "PingFang TC", "Heiti TC", "Microsoft JhengHei", "sans-serif"]
plt.rcParams["axes.unicode_minus"] = False

print("環境與繪圖中文字型設定完成！")

## A1 讀取與觀察資料（EDA：探索性資料分析）
讀取 `data/contract_risk.csv`，檢視前 5 筆資料、欄位類型、缺失值，以及合約風險分佈。

In [ ]:
# 讀取合約歷史資料集
df = pd.read_csv("data/contract_risk.csv")
print(f"資料筆數: {df.shape[0]} 筆，欄位數: {df.shape[1]} 欄\n")
print("前五筆資料:")
display(df.head())

print("\n缺失值統計:")
print(df.isnull().sum())

print("\n目標標籤 risk_level 分佈:")
print(df["risk_level"].value_counts())

# 繪製目標標籤分佈長條圖
plt.figure(figsize=(7, 4))
df["risk_level"].value_counts().plot(kind="bar", color=["#dd6b20", "#2f855a", "#c53030"])
plt.title("歷史合約法律風險等級分佈")
plt.xlabel("風險等級")
plt.ylabel("合約筆數")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

## A2 切分特徵（X）與標籤（y），保留測試集
- 剔除純識別識別碼 `contract_id`
- 以 `risk_level` 為目標標籤 y，其餘欄位為輸入特徵 X
- 採用 8:2 切分訓練集（Train Set）與測試集（Test Set），使用 `stratify=y` 確保各類別比例一致

In [ ]:
from sklearn.model_selection import train_test_split

# 定義特徵 X 與目標 y
X = df.drop(columns=["contract_id", "risk_level"])
y = df["risk_level"]

# 80% 訓練集，20% 測試集（神聖不可侵犯，嚴禁洩漏）
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"訓練集大小: {X_train.shape[0]} 筆")
print(f"測試集大小: {X_test.shape[0]} 筆")

## A3 建立前處理 Pipeline 與隨機森林模型訓練
- 類別特徵（`contract_type`）：使用 `OneHotEncoder`
- 數值特徵：使用 `StandardScaler` 標準化
- 模型：`RandomForestClassifier`（隨機森林分類器，適合處理條款特徵與數值非線性關係）
- 使用 5 折交叉驗證（Cross-Validation）檢視模型穩定度

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.model_selection import cross_val_score

cat_features = ["contract_type"]
num_features = [c for c in X.columns if c != "contract_type"]

preprocessor = ColumnTransformer([
    ("cat", OneHotEncoder(handle_unknown="ignore"), cat_features),
    ("num", StandardScaler(), num_features)
])

pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", RandomForestClassifier(n_estimators=100, max_depth=8, random_state=42))
])

# 5 折交叉驗證評估
cv_scores = cross_val_score(pipeline, X_train, y_train, cv=5, scoring="accuracy")
print(f"5 折交叉驗證平均準確率: {cv_scores.mean():.4f} (±{cv_scores.std():.4f})")

# 在完整訓練集上進行擬合
pipeline.fit(X_train, y_train)
print("模型訓練完成！")

## A4 測試集驗收、混淆矩陣與特徵重要性分析
- 以先前完全未看過的 `X_test` 進行最終預測
- 檢視 Precision（精確率）、Recall（召回率）、F1-Score
- 繪製混淆矩陣，並檢視法務最在意的「特徵重要性（Feature Importance）」

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay

# 測試集推論
y_pred = pipeline.predict(X_test)

print("=== 測試集分類評估報告 ===")
print(classification_report(y_test, y_pred))

# 繪製混淆矩陣
fig, ax = plt.subplots(figsize=(6, 5))
labels = ["低風險", "中風險", "高風險"]
cm = confusion_matrix(y_test, y_pred, labels=labels)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=labels)
disp.plot(cmap="Blues", ax=ax)
plt.title("合約風險預測混淆矩陣")
plt.tight_layout()
plt.show()

# 分析影響合約風險的前幾大特徵
rf = pipeline.named_steps["classifier"]
ohe = pipeline.named_steps["preprocessor"].named_transformers_["cat"]
ohe_cols = list(ohe.get_feature_names_out(cat_features))
all_feature_names = ohe_cols + num_features
importances = pd.Series(rf.feature_importances_, index=all_feature_names).sort_values(ascending=False)

plt.figure(figsize=(8, 4))
importances.head(8).plot(kind="barh", color="#3182ce").invert_yaxis()
plt.title("法務合約風險特徵重要性排行（Top 8）")
plt.xlabel("重要性權重")
plt.tight_layout()
plt.show()

## A5 保存模型與元資料
將訓練好的 Pipeline 與欄位清單保存至 `artifacts/` 目錄，以便 `app.py` 桌面視窗程式直接讀取！

In [ ]:
import json
import joblib
from sklearn.metrics import accuracy_score

os.makedirs("artifacts", exist_ok=True)

# 1. 保存完整 Pipeline 模型
joblib.dump(pipeline, "artifacts/model.joblib")

# 2. 保存元資料 JSON
metadata = {
    "task": "multiclass_classification",
    "target": "risk_level",
    "classes": list(pipeline.classes_),
    "contract_types": sorted(df["contract_type"].unique().tolist()),
    "test_accuracy": round(float(accuracy_score(y_test, y_pred)), 4),
    "samples_count": len(df)
}

with open("artifacts/metadata.json", "w", encoding="utf-8") as f:
    json.dump(metadata, f, ensure_ascii=False, indent=2)

print("model.joblib 與 metadata.json 已成功儲存至 artifacts/ 目錄！")
print("現在您可以前往終端機執行: uv run python app.py 啟動 Tkinter 桌面預測視窗！")